# Computing $\alpha^\beta$ via Bounded GPAC

This notebook simulates the bounded GPAC construction for computing $\alpha^\beta$ from upstream analog computations of $\alpha$ and $\beta$.

**Construction:** Given upstream GPACs computing $x(t) \to \alpha$ and $y(t) \to \beta$:

$$\begin{aligned}
x_1' &= (x - 1) - x_1, \\
u' &= (1 - v)\,x_1', \\
v' &= (1 - v)^2\,x_1', \\
z' &= z\,(y'\,u + y\,(1-v)\,x_1'),
\end{aligned}$$

with $x_1(0) = u(0) = v(0) = 0$, $z(0) = 1$.

**Key identity:** $z(t) = e^{y(t) \cdot u(t)} \to e^{\beta \ln \alpha} = \alpha^\beta$.

**Complexity:** $\mu_{\alpha^\beta}(r) = \max(\mu_\alpha(r+C), \mu_\beta(r+C)) + O(1)$.

Blog post: [How Hard Is It to Compute α^β?](https://infsup.com/math/alpha-beta-complexity/)

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [ ]:
def simulate_alpha_beta(alpha_func, alpha_deriv, beta_func, beta_deriv,
                        alpha_star, beta_star, T=20):
    """
    Simulate the α^β bounded GPAC construction.
    
    Parameters
    ----------
    alpha_func, alpha_deriv : callable  — x(t) → α and x'(t)
    beta_func, beta_deriv : callable    — y(t) → β and y'(t)
    alpha_star, beta_star : float       — target values
    T : float                           — simulation time
    """
    target = alpha_star ** beta_star

    def rhs(t, state):
        x1, u, v, z = state
        x_t = alpha_func(t)
        y_t = beta_func(t)
        dy = beta_deriv(t)

        x1_dot = (x_t - 1) - x1
        u_dot = (1 - v) * x1_dot
        v_dot = (1 - v)**2 * x1_dot
        z_dot = z * (dy * u + y_t * (1 - v) * x1_dot)

        return [x1_dot, u_dot, v_dot, z_dot]

    sol = solve_ivp(rhs, [0.001, T], [0, 0, 0, 1],
                    max_step=0.005, dense_output=True,
                    rtol=1e-12, atol=1e-14)
    t = np.linspace(0.001, T, 3000)
    y = sol.sol(t)
    return t, y[0], y[1], y[2], y[3], target

## Example 1: $(\pi/4)^e \approx 0.5186$

Both $\pi/4$ and $e$ are computed by upstream GPACs with exponential convergence (linear time).

In [ ]:
# Input GPACs: exponential convergence to π/4 and e
pi4_func  = lambda t: np.pi/4 * (1 - np.exp(-t))
pi4_deriv = lambda t: np.pi/4 * np.exp(-t)
e_func    = lambda t: np.e * (1 - np.exp(-t))
e_deriv   = lambda t: np.e * np.exp(-t)

t, x1, u, v, z, target = simulate_alpha_beta(
    pi4_func, pi4_deriv, e_func, e_deriv, np.pi/4, np.e, T=15)

print(f'Target: (π/4)^e = {target:.8f}')
print(f'z(T)           = {z[-1]:.8f}')
print(f'Error          = {abs(z[-1] - target):.2e}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

# (a) Inputs
ax = axes[0, 0]
ax.plot(t, pi4_func(t), 'C0', lw=2, label='α(t) → π/4')
ax.plot(t, e_func(t), 'C1', lw=2, label='β(t) → e')
ax.axhline(y=np.pi/4, color='C0', ls='--', alpha=0.3)
ax.axhline(y=np.e, color='C1', ls='--', alpha=0.3)
ax.set_xlabel('Time t'); ax.set_title('(a) Inputs'); ax.legend()

# (b) Internal variables
ax = axes[0, 1]
ax.plot(t, x1, 'C2', lw=2, label=f'x₁ → {np.pi/4-1:.4f}')
ax.plot(t, u, 'C3', lw=2, label=f'u → ln(π/4) = {np.log(np.pi/4):.4f}')
ax.plot(t, v, 'C4', lw=2, label=f'v → {(np.pi/4-1)/(np.pi/4):.4f}')
ax.set_xlabel('Time t'); ax.set_title('(b) Internal variables'); ax.legend(fontsize=8)

# (c) Output
ax = axes[1, 0]
ax.plot(t, z, 'C3', lw=2.5, label=f'z(t) → (π/4)^e ≈ {target:.4f}')
ax.axhline(y=target, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Time t'); ax.set_ylabel('z(t)'); ax.set_title('(c) Output'); ax.legend()

# (d) Error
ax = axes[1, 1]
err = np.abs(z - target)
ax.semilogy(t, np.maximum(err, 1e-15), 'C3', lw=2)
ax.set_xlabel('Time t'); ax.set_ylabel('|z(t) - (π/4)^e|')
ax.set_title('(d) Error (log scale)'); ax.grid(True, alpha=0.3)

plt.suptitle('Computing (π/4)^e via Bounded GPAC', fontsize=14)
plt.tight_layout()
plt.show()

## Example 2: $e^\pi \approx 23.14$

In [ ]:
pi_func  = lambda t: np.pi * (1 - np.exp(-t))
pi_deriv = lambda t: np.pi * np.exp(-t)

t2, x1_2, u_2, v_2, z_2, target2 = simulate_alpha_beta(
    e_func, e_deriv, pi_func, pi_deriv, np.e, np.pi, T=15)

print(f'Target: e^π = {target2:.8f}')
print(f'z(T)        = {z_2[-1]:.8f}')
print(f'Error       = {abs(z_2[-1] - target2):.2e}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

ax = axes[0, 0]
ax.plot(t2, e_func(t2), 'C0', lw=2, label='α(t) → e')
ax.plot(t2, pi_func(t2), 'C1', lw=2, label='β(t) → π')
ax.axhline(y=np.e, color='C0', ls='--', alpha=0.3)
ax.axhline(y=np.pi, color='C1', ls='--', alpha=0.3)
ax.set_xlabel('Time t'); ax.set_title('(a) Inputs'); ax.legend()

ax = axes[0, 1]
ax.plot(t2, x1_2, 'C2', lw=2, label=f'x₁ → e-1 = {np.e-1:.4f}')
ax.plot(t2, u_2, 'C3', lw=2, label='u → ln(e) = 1.0000')
ax.plot(t2, v_2, 'C4', lw=2, label=f'v → (e-1)/e = {(np.e-1)/np.e:.4f}')
ax.set_xlabel('Time t'); ax.set_title('(b) Internal variables'); ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(t2, z_2, 'C0', lw=2.5, label=f'z(t) → e^π ≈ {target2:.4f}')
ax.axhline(y=target2, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Time t'); ax.set_ylabel('z(t)'); ax.set_title('(c) Output'); ax.legend()

ax = axes[1, 1]
err2 = np.abs(z_2 - target2)
ax.semilogy(t2, np.maximum(err2, 1e-15), 'C0', lw=2)
ax.set_xlabel('Time t'); ax.set_ylabel('|z(t) - e^π|')
ax.set_title('(d) Error (log scale)'); ax.grid(True, alpha=0.3)

plt.suptitle('Computing e^π via Bounded GPAC', fontsize=14)
plt.tight_layout()
plt.show()

## Try Your Own!

Change `alpha_star` and `beta_star` below to compute any $\alpha^\beta$.

In [ ]:
# === CUSTOMIZE HERE ===
alpha_star = 2.0
beta_star = 0.5   # Should compute √2 ≈ 1.41421
T = 15
# ======================

# Simple exponential-convergence inputs
af = lambda t: alpha_star * (1 - np.exp(-t))
ad = lambda t: alpha_star * np.exp(-t)
bf = lambda t: beta_star * (1 - np.exp(-t))
bd = lambda t: beta_star * np.exp(-t)

t, x1, u, v, z, tgt = simulate_alpha_beta(af, ad, bf, bd, alpha_star, beta_star, T)

print(f'Computing {alpha_star}^{beta_star} = {tgt:.8f}')
print(f'z(T) = {z[-1]:.8f}, error = {abs(z[-1]-tgt):.2e}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(t, z, 'C3', lw=2)
ax1.axhline(y=tgt, color='gray', ls='--', alpha=0.5)
ax1.set_title(f'z(t) → {alpha_star}^{{{beta_star}}} = {tgt:.4f}')
ax1.set_xlabel('Time t')

ax2.semilogy(t, np.maximum(np.abs(z - tgt), 1e-15), 'C3', lw=2)
ax2.set_title('Error (log scale)')
ax2.set_xlabel('Time t'); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()